# Antes de implementar RAG
Ejemplo de como responde el modelo antes de implementar la arquitectura RAG

In [18]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Inicializar el modelo
model = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash-lite",
    temperature = 0,
)

# Le decimos al modelo lo que queremos que haga
messages = [
    (
        "system",
        """Eres un asesor financiero que responde preguntas de los usuarios con explicaciones
        adecuadas y concisas para quienes no estan acostumbrados al lenguaje de las finanzas. 
        Tus respuestas deben estar en el idioma en el que fue hecha la pregunta""",
    ),
    ("human", "Cuales son los pasos a seguir para empezar a invertir?"),
]

# Imprimir la respuesta del modelo
msg = model.invoke(messages)
print(msg.content)

¡Excelente pregunta! Empezar a invertir puede parecer complicado al principio, pero si sigues estos pasos, verás que es más sencillo de lo que crees:

1.  **Define tus objetivos:** Antes de poner tu dinero en cualquier lugar, piensa para qué quieres invertir. ¿Es para tu jubilación, para comprar una casa en unos años, para tener un colchón de emergencia o simplemente para que tu dinero crezca? Saber esto te ayudará a elegir las inversiones adecuadas.

2.  **Evalúa tu situación financiera:**
    *   **¿Tienes deudas?** Si tienes deudas con intereses altos (como tarjetas de crédito), a menudo es mejor pagarlas primero, ya que el interés que pagas puede ser mayor que el rendimiento que obtienes invirtiendo.
    *   **¿Tienes un fondo de emergencia?** Es crucial tener dinero ahorrado para imprevistos (pérdida de empleo, gastos médicos inesperados). Este dinero debe estar en un lugar seguro y de fácil acceso, no invertido.

3.  **Edúcate sobre las opciones de inversión:** Hay muchos lugares

# Implementacion RAG

### Document loading

In [1]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

path = Path("./db/books")
allDocs = []
# Iteramos cada archivo en la ruta
for file in path.iterdir():
    loader = PyPDFLoader(file)
    pages = loader.load()
    # Interceptamos el archivo para modificar su metadata
    for page in pages:
        if file.name == "Padre_Pobre_Padre_Rico.pdf":
            page.metadata['title'] = 'Padre Rico, Padre Pobre'
            page.metadata['author'] = 'Robert T. Kiyosaki, Sharon L. Lechter' 
            page.metadata['subject'] = 'Finanzas Personales'
    # print(pages[0].metadata)
    allDocs.extend(pages)
    print(f"File {file.name} loaded. Pages: {len(pages)}")

File EL INVERSOR INTELIGENTE - BENJAMIN GRAHAM.pdf loaded. Pages: 1044
File el-pequeno-libro-para-invertir-john-c-bogle.pdf loaded. Pages: 165
File El_hombre_mas_rico_de_Babilonia_George_S_Clason.pdf loaded. Pages: 121
File Los secretos de la mente millonaria - T. Harv Eker.pdf loaded. Pages: 140
File Padre_Pobre_Padre_Rico.pdf loaded. Pages: 231
File Pequeño Cerdo Capitalista.pdf loaded. Pages: 150
File The-Psychology-of-Money-Morgan-Housel.pdf loaded. Pages: 242


### Document splitting

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunkSize = 1200
chunkOverlap = 200

rSplitter = RecursiveCharacterTextSplitter(
    chunk_size = chunkSize,
    chunk_overlap = chunkOverlap
)

splittedDocs = rSplitter.split_documents(allDocs)
print(f"Páginas originales: {len(allDocs)}")
print(f"Chunks generados listos para la base de datos: {len(splittedDocs)}")

Páginas originales: 2093
Chunks generados listos para la base de datos: 4192


### Embeddings y Vectorstores

#### Prueba para medir el tiempo que tarda el cpu en generar los embeddings

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

# Descarga y carga del modelo Google EmbeddingGemma-300m en la GPU
opciones_modelo = {'device': 'cpu'}

embedding = HuggingFaceEmbeddings(
    model_name="google/embeddinggemma-300m",
    model_kwargs=opciones_modelo
)

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

In [ ]:
%%time
from langchain_chroma import Chroma

persistDir = "./db/vectorial-db-cpu"

# Inicializacion de una chroma db
chromaDb = Chroma.from_documents(
    documents=splittedDocs,
    embedding=embedding,
    persist_directory=persistDir
)

print(chromaDb._collection.count())

Finished building Vector Database!
Total collections in DB: 4192
4192
CPU times: total: 2h 57min 20s
Wall time: 22min 41s


#### Generacion de los embeddings con la gpu

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

# Descarga y carga del modelo Google EmbeddingGemma-300m en la GPU
opciones_modelo = {'device': 'cuda'}

embedding = HuggingFaceEmbeddings(
    model_name="google/embeddinggemma-300m",
    model_kwargs=opciones_modelo
)

print("Modelo cargado en la tarjeta gráfica")

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

Modelo cargado en la tarjeta gráfica


In [5]:
from sklearn.metrics.pairwise import cosine_similarity

# Probamos que tal funciona el modelo de embedding con diferentes idiomas
vectorEs = embedding.embed_query("¿Qué es un fondo indexado?")
vectorEn = embedding.embed_query("What is an index fund?")
vectorGe = embedding.embed_query("Was ist ein Indexfonds?")

print(f"Dimension de los vectores: {len(vectorEs)}")

print(f"Cosine similarity: {cosine_similarity([vectorEs], [vectorEn])[0][0]}")
print(f"Cosine similarity: {cosine_similarity([vectorEs], [vectorGe])[0][0]}")
print(f"Cosine similarity: {cosine_similarity([vectorEn], [vectorGe])[0][0]}")

Dimension de los vectores: 768
Cosine similarity: 0.7704835636858518
Cosine similarity: 0.8467191784612165
Cosine similarity: 0.9260788299084759


In [6]:
from langchain_chroma import Chroma

persistDir = "./db/vectorial-db"

In [7]:
%%time

# Creacion de una chroma db persistente
chromaDb = Chroma.from_documents(
    documents=splittedDocs,
    embedding=embedding,
    persist_directory=persistDir
)

print(f"Total de colecciones en la DB: {chromaDb._collection.count()}")

Total de colecciones en la DB: 4192
CPU times: total: 5min 11s
Wall time: 3min 11s


### Retrieval

In [ ]:
import google.generativeai as genai

# Peticion a la API que liste todos los modelos disponibles
print("Modelos disponibles:")
for m in genai.list_models():
    # Filtramos para ver solo los que sirven para generar texto
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

In [14]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

llm = ChatGoogleGenerativeAI(
    model = "gemma-3-27b-it",
    temperature = 0,
)

# Recuperacion de la DB
chromaDB = Chroma(
    persist_directory=persistDir,
    embedding_function=embedding
)

# Uso del mismo modelo para el compresor
compressor = LLMChainExtractor.from_llm(llm)

# Uso de mmr como base para tener variedad
baseRetriever = chromaDB.as_retriever(
    search_type = "mmr",
    search_kwargs = {"k": 4, "fetch_k": 25}
)

compressedRetriever = ContextualCompressionRetriever(
    base_retriever=baseRetriever,
    base_compressor=compressor
)


In [15]:
def pretty_print_docs(docs):
    print(f"\n{'-' * 100}\n".join([f"Document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]))

# Probamos como funciona el retriever
question = "Cuales son los pasos a seguir para empezar a invertir?"
compressedDocs = compressedRetriever.invoke(question)
pretty_print_docs(compressedDocs)

Document 1:

Lo más recomendable es que empieces con instrumentos menos riesgosos, investigues un poco más del mercado bursátil y cuando te sientas más seguro vayas metiendo poco a poco un porcentaje de tu portafolio a la Bolsa (empieza con 10% máximo), para que una mala racha no te decepcione y te cure de espanto para siempre.


### Question answering

In [21]:
# Funcion para formatear la lista de documentos a texto plano
def format_docs(docs):
    text_chunks = []
    for doc in docs:
        # Extraemos los metadatos con valores por defecto por si alguno falla
        autor = doc.metadata.get('author', 'Autor Desconocido')
        libro = doc.metadata.get('title', 'Libro Desconocido')
        contenido = doc.page_content
        
        # Le damos un formato claro para que el LLM sepa quién fue el autor
        chunk_formateado = f"[Fuente: {libro}, Autor: {autor}]\n{contenido}"
        text_chunks.append(chunk_formateado)
        
    return "\n\n---\n\n".join(text_chunks)

In [22]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Inicializa el parser para convertir la respuesta del modelo en una cadena de texto simple
outputParser = StrOutputParser()

# Define la estructura del prompt, incluyendo el rol del sistema y las variables de entrada
prompt = ChatPromptTemplate.from_messages([
    ("system", """
    Eres un asesor financiero experto y servicial que responde preguntas basándose estrictamente en el texto de referencia proporcionado.
    Asegúrate de ser exhaustivo y proporcionar la información de fondo relevante.
    Estás hablando con una audiencia no técnica, por lo que debes desglosar los conceptos complicados y mantener un tono amigable.
    CRÍTICO: Cuando utilices información del texto de referencia, SIEMPRE menciona al autor y el libro del que proviene el consejo.
    Si la respuesta no está en el texto de referencia, di amablemente que no tienes esa información.
    Por favor, responde en el idioma de la pregunta.
    """),
    ("human", """
    PREGUNTA: '{query}'
    
    TEXTO DE REFERENCIA:
    {reference}
    
    RESPUESTA:
    """
    ),
])

# Construye la cadena (chain) vinculando el prompt, el modelo y el parser de salida
chain = prompt | model | outputParser

In [23]:
query = "Cuales son los pasos a seguir para empezar a invertir?"

# Recuperacion de los documentos el retriever
retrievedDocs = compressedRetriever.invoke(query)

# Formateo de los documentos
reference = format_docs(retrievedDocs)

# Ejecuta la cadena con la consulta del usuario y la referencia
answer = chain.invoke({
    "query": query,
    "reference": reference,
})

print(answer)

¡Hola! Para empezar a invertir, Sofía Macías, autora de "Pequeño cerdo capitalista", recomienda un enfoque gradual y prudente.

Según ella, lo más aconsejable es **comenzar con instrumentos que tengan menos riesgo**. Una vez que te sientas más cómodo y hayas investigado un poco sobre el mercado bursátil, puedes empezar a **invertir gradualmente un pequeño porcentaje de tu dinero en la Bolsa**.

Macías sugiere que este porcentaje inicial **no supere el 10% de tu portafolio**. La idea detrás de esto es que si el mercado tiene un mal desempeño, no te desanimes ni te alejes de la inversión para siempre. De esta manera, puedes ir aprendiendo y ganando confianza sin poner en riesgo una gran parte de tus ahorros.
